In [14]:
# ============================================================
# 라이브러리 및 경로 설정
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_DIR = Path.cwd().parents[2]

DATA_DIR = PROJECT_DIR / "data"
V2_DATA_DIR = DATA_DIR / "v2_5year_seoul"
PROCESSED_DIR = V2_DATA_DIR / "processed"


base = pd.read_parquet(
    PROCESSED_DIR / "df_fe_base.parquet"
)

school = pd.read_parquet(
    PROCESSED_DIR / "df_fe_school.parquet"
)

subway = pd.read_parquet(
    PROCESSED_DIR / "df_fe_subway.parquet"
)

living = pd.read_parquet(
    PROCESSED_DIR / "df_fe_living.parquet"
)

rent = pd.read_parquet(
    PROCESSED_DIR / "df_fe_rent.parquet"
)

rate = pd.read_parquet(
    PROCESSED_DIR / "df_fe_rate.parquet"
)

policy = pd.read_parquet(
    PROCESSED_DIR / "df_fe_policy.parquet"
)

In [15]:
# ============================================================
# 데이터 구조 확인
# ============================================================

datasets = {
    "base": base,
    "school": school,
    "subway": subway,
    "living": living,
    "rent": rent,
    "rate": rate,
    "policy": policy,
}

for name, df in datasets.items():
    print(
        name,
        df.shape,
        "| id 중복:",
        df["transaction_id"].duplicated().sum(),
    )

base (234302, 69) | id 중복: 0
school (234302, 9) | id 중복: 0
subway (234302, 5) | id 중복: 0
living (234302, 12) | id 중복: 0
rent (234302, 19) | id 중복: 0
rate (234302, 10) | id 중복: 0
policy (234302, 6) | id 중복: 0


In [16]:
# ============================================================
# Feature명 중복 검사
# ============================================================

feature_datasets = {
    "school": school,
    "subway": subway,
    "living": living,
    "rent": rent,
    "rate": rate,
    "policy": policy,
}

base_cols = set(base.columns)

for name, df in feature_datasets.items():
    overlap = (
        base_cols
        & set(df.columns)
        - {"transaction_id"}
    )

    print(
        name,
        "중복 컬럼:",
        sorted(overlap),
    )

school 중복 컬럼: []
subway 중복 컬럼: []
living 중복 컬럼: []
rent 중복 컬럼: []
rate 중복 컬럼: []
policy 중복 컬럼: []


In [17]:
# ============================================================
# 전체 Feature 통합
# ============================================================

df_final = base.copy()

for name, feature_df in feature_datasets.items():

    before_rows = len(df_final)

    df_final = df_final.merge(
        feature_df,
        on="transaction_id",
        how="left",
        validate="one_to_one",
    )

    print(
        f"{name:8s}",
        f"{before_rows:,}",
        "→",
        f"{len(df_final):,}",
        "| columns:",
        df_final.shape[1],
    )

school   234,302 → 234,302 | columns: 77
subway   234,302 → 234,302 | columns: 81
living   234,302 → 234,302 | columns: 92
rent     234,302 → 234,302 | columns: 110
rate     234,302 → 234,302 | columns: 119
policy   234,302 → 234,302 | columns: 124


In [18]:
# ============================================================
# 최종 통합 데이터 검증
# ============================================================

print("최종 shape:", df_final.shape)

print(
    "transaction_id 중복:",
    df_final["transaction_id"]
    .duplicated()
    .sum(),
)

print(
    "transaction_id 결측:",
    df_final["transaction_id"]
    .isna()
    .sum(),
)

최종 shape: (234302, 124)
transaction_id 중복: 0
transaction_id 결측: 0


In [19]:
# ============================================================
# 모듈별 결측치 확인
# ============================================================

for name, feature_df in feature_datasets.items():

    feature_cols = [
        col
        for col in feature_df.columns
        if col != "transaction_id"
    ]

    missing = (
        df_final[feature_cols]
        .isna()
        .sum()
        .sum()
    )

    print(
        f"{name:8s}",
        "결측:",
        f"{missing:,}",
    )

school   결측: 0
subway   결측: 0
living   결측: 0
rent     결측: 24,841
rate     결측: 0
policy   결측: 0


In [20]:
# ============================================================
# 전체 결측 컬럼 확인
# ============================================================

missing_summary = (
    df_final
    .isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print(
    "결측 존재 컬럼 수:",
    len(missing_summary),
)

display(
    missing_summary.to_frame(
        name="결측수"
    )
)

결측 존재 컬럼 수: 34


,결측수
해제여부,234302
해제사유발생일,234302
매수자,62194
매도자,62194
아파트동명,55559
등기일자,39310
도로명지상지하코드,26899
중개사소재지,25599
구_과거6개월_거래량,16113
서울_과거6개월_중위가격,16113


In [24]:
# ============================================================
# 최종 결측 구조 검증
# ============================================================

missing_summary = (
    df_final
    .isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

display(
    missing_summary.to_frame("결측수")
)

,결측수
해제여부,234302
해제사유발생일,234302
매수자,62194
매도자,62194
아파트동명,55559
등기일자,39310
도로명지상지하코드,26899
중개사소재지,25599
구_과거6개월_거래량,16113
서울_과거6개월_중위가격,16113


In [25]:
# ============================================================
# 전세가율 구조적 결측 기간 검증
# ============================================================

ratio_cols = [
    "전세가율_전월",
    "전세가율_과거3개월",
    "전세가율_과거6개월",
]

ratio_missing = (
    df_final
    .groupby("거래월일자")[ratio_cols]
    .apply(lambda x: x.isna().sum())
)

display(
    ratio_missing[
        ratio_missing.sum(axis=1) > 0
    ]
)

,전세가율_전월,전세가율_과거3개월,전세가율_과거6개월
거래월일자,,,
2021-07-01,4677,4677,4677
2021-08-01,0,4060,4060
2021-09-01,0,0,2690


In [26]:
# ============================================================
# 최종 V1 데이터 저장
# ============================================================

FINAL_PATH = (
    PROCESSED_DIR
    / "df_final_v1.parquet"
)

df_final.to_parquet(
    FINAL_PATH,
    index=False,
)

print("저장 완료:", FINAL_PATH)
print("shape:", df_final.shape)

print(
    "transaction_id 중복:",
    df_final["transaction_id"]
    .duplicated()
    .sum(),
)

저장 완료: c:\Users\uk\Apartment price\data\v2_5year_seoul\processed\df_final_v1.parquet
shape: (234302, 124)
transaction_id 중복: 0
